# Lab 7 - ANN dự đoán giá xe hơi



## Bước 1: Import thư viện

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

## Bước 2: Đọc dữ liệu từ GitHub

In [ ]:
url = "https://raw.githubusercontent.com/huynhhoc/DataAnalystDeepLearning/main/Data/carpricesdata.csv"

df = pd.read_csv(url)

print("Đọc dữ liệu thành công!")
print("Kích thước dữ liệu:", df.shape)
df.head()

## Bước 3: Kiểm tra dữ liệu

In [ ]:
print("Thông tin dữ liệu:")
df.info()

print("\nSố lượng giá trị thiếu ở từng cột:")
print(df.isnull().sum())

print("\nThống kê mô tả:")
df.describe()

## Bước 4: Chọn biến đầu vào X và biến mục tiêu y

In [ ]:
# Các cột dùng để dự đoán giá xe
predictors = ["Age", "KM", "Weight", "HP", "MetColor", "CC", "Doors"]

# Cột cần dự đoán
target = "Price"

X = df[predictors]
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

## Bước 5: Chia tập train/test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Số mẫu train:", X_train.shape[0])
print("Số mẫu test :", X_test.shape[0])

## Bước 6: Chuẩn hóa dữ liệu

In [ ]:
# Chuẩn hóa input X
x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

# Chuẩn hóa output y để model học ổn định hơn
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1)).ravel()

print("Đã chuẩn hóa dữ liệu xong.")
print("X_train_scaled shape:", X_train_scaled.shape)
print("y_train_scaled shape:", y_train_scaled.shape)

## Bước 7: Xây dựng mô hình ANN

In [ ]:
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.1),

    Dense(32, activation="relu"),
    Dropout(0.1),

    Dense(16, activation="relu"),

    # Output cho bài toán hồi quy: 1 neuron, activation linear
    Dense(1, activation="linear")
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

## Bước 8: Huấn luyện mô hình

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

## Bước 9: Vẽ biểu đồ Loss và MAE

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Biểu đồ Loss qua từng epoch")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history.history["mae"], label="Train MAE")
plt.plot(history.history["val_mae"], label="Validation MAE")
plt.title("Biểu đồ MAE qua từng epoch")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()
plt.grid(True)
plt.show()

## Bước 10: Dự đoán trên tập test

In [ ]:
# Dự đoán giá đã chuẩn hóa
y_pred_scaled = model.predict(X_test_scaled)

# Đưa giá trị dự đoán về đơn vị giá thật
y_pred = y_scaler.inverse_transform(y_pred_scaled).ravel()

# y thực tế
y_true = y_test.values

result_df = pd.DataFrame({
    "Giá thực tế": y_true,
    "Giá dự đoán": y_pred,
    "Sai số": y_pred - y_true,
    "Sai số tuyệt đối": np.abs(y_pred - y_true)
})

result_df.head(10)

## Bước 11: Đánh giá mô hình

In [ ]:
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ANN")
print("--------------------------------")
print(f"MAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:.4f}")

## Bước 12: Vẽ biểu đồ giá thực tế và giá dự đoán

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_true, y_pred, alpha=0.7)

min_value = min(y_true.min(), y_pred.min())
max_value = max(y_true.max(), y_pred.max())

plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.xlabel("Giá thực tế")
plt.ylabel("Giá dự đoán")
plt.title("So sánh giá thực tế và giá dự đoán")
plt.grid(True)
plt.show()

## Bước 13: Dự đoán thử một xe mới

In [ ]:
# Ví dụ xe mới:
# Age=36 tháng, KM=50000, Weight=1100, HP=110, MetColor=1, CC=1600, Doors=4
new_car = pd.DataFrame([{
    "Age": 36,
    "KM": 50000,
    "Weight": 1100,
    "HP": 110,
    "MetColor": 1,
    "CC": 1600,
    "Doors": 4
}])

new_car_scaled = x_scaler.transform(new_car)
new_price_scaled = model.predict(new_car_scaled)
new_price = y_scaler.inverse_transform(new_price_scaled).ravel()[0]

print(f"Giá xe dự đoán: {new_price:,.2f}")

## Bước 14: Lưu kết quả

In [ ]:
result_df.to_csv("ket_qua_du_doan_gia_xe_ANN.csv", index=False)

model.save("model_ANN_du_doan_gia_xe.keras")

print("Đã lưu:")
print("- ket_qua_du_doan_gia_xe_ANN.csv")
print("- model_ANN_du_doan_gia_xe.keras")

Test bằng dữ liệu trên https://www.kaggle.com/datasets/victorahaji/toyota-corolla-car-price-prediction/data

BƯỚC 1: Cài thư viện KaggleHub


In [ ]:
!pip install kagglehub[pandas-datasets]

BƯỚC 2: Import thư viện cần dùng

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

BƯỚC 3: Tải dataset Toyota Corolla từ Kaggle

In [ ]:
path = kagglehub.dataset_download("victorahaji/toyota-corolla-car-price-prediction")

print("Dataset được tải về tại:", path)
print("Danh sách file trong dataset:")
print(os.listdir(path))

BƯỚC 4: Đọc file CSV

In [ ]:
csv_files = [file for file in os.listdir(path) if file.endswith(".csv")]

print("Các file CSV tìm thấy:", csv_files)

file_path = os.path.join(path, csv_files[0])
df = pd.read_csv(file_path)

print("Kích thước dữ liệu ban đầu:", df.shape)
df.head()

BƯỚC 5: Kiểm tra tên cột trong dataset Kaggle

In [ ]:
print("Danh sách cột trong dataset:")
print(df.columns.tolist())

BƯỚC 6: Lọc dữ liệu cho giống bài thầy

In [ ]:
# Copy dữ liệu gốc để xử lý
data = df.copy()

# Đổi tên cột cho giống bài thầy nếu dataset Kaggle đặt tên khác
rename_dict = {}

if 'Age_08_04' in data.columns:
    rename_dict['Age_08_04'] = 'Age'

if 'Met_Color' in data.columns:
    rename_dict['Met_Color'] = 'MetColor'

if 'cc' in data.columns:
    rename_dict['cc'] = 'CC'

data = data.rename(columns=rename_dict)

print("Danh sách cột sau khi đổi tên:")
print(data.columns.tolist())

BƯỚC 7: Chỉ giữ lại các cột

In [ ]:
lab_columns = ['Age', 'KM', 'Weight', 'HP', 'MetColor', 'CC', 'Doors', 'Price']

lab_data = data[lab_columns].copy()

print("Kích thước dữ liệu sau khi lọc:", lab_data.shape)
lab_data.head()

BƯỚC 8: Kiểm tra dữ liệu bị thiếu

In [ ]:
print("Số lượng giá trị thiếu ở từng cột:")
print(lab_data.isnull().sum())

BƯỚC 9: Kiểm tra kiểu dữ liệu

In [ ]:
lab_data.info()

BƯỚC 10: Ép kiểu dữ liệu về số

In [ ]:
for col in lab_columns:
    lab_data[col] = pd.to_numeric(lab_data[col], errors='coerce')

lab_data = lab_data.dropna()

print("Kích thước dữ liệu sau khi ép kiểu số:", lab_data.shape)
lab_data.head()

BƯỚC 11: Xem thống kê dữ liệu

In [ ]:
lab_data.describe()

BƯỚC 12: Tách input và output

In [ ]:
Predictors = ['Age', 'KM', 'Weight', 'HP', 'MetColor', 'CC', 'Doors']
TargetVariable = ['Price']

X = lab_data[Predictors].values
y = lab_data[TargetVariable].values

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)

BƯỚC 13: Chia dữ liệu train/test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Số dòng train:", X_train.shape[0])
print("Số dòng test:", X_test.shape[0])

BƯỚC 14: Chuẩn hóa dữ liệu

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("y_train_scaled shape:", y_train_scaled.shape)

BƯỚC 15: Xây dựng mô hình ANN

In [ ]:
model = Sequential()

model.add(Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))

# Output 1 neuron để dự đoán Price
model.add(Dense(1, activation='linear'))

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mae']
)

model.summary()

In [ ]:
BƯỚC 16: Train mô hình ANN

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=30,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_data=(X_test_scaled, y_test_scaled),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

BƯỚC 17: Vẽ biểu đồ Loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Biểu đồ Loss của mô hình ANN')
plt.legend()
plt.grid(True)
plt.show()

BƯỚC 18: Vẽ biểu đồ MAE

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.title('Biểu đồ MAE của mô hình ANN')
plt.legend()
plt.grid(True)
plt.show()

BƯỚC 19: Dự đoán trên tập test

In [ ]:
y_pred_scaled = model.predict(X_test_scaled)

# Chuyển kết quả dự đoán về giá trị Price thật
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# y_test ban đầu đã là giá thật
y_test_real = y_test

print("5 giá trị dự đoán đầu tiên:")
print(y_pred[:5])

print("5 giá trị thực tế đầu tiên:")
print(y_test_real[:5])

Tạo bảng so sánh giá thật và giá dự đoán

In [ ]:
result = pd.DataFrame({
    'Gia_thuc_te': y_test_real.flatten(),
    'Gia_du_doan': y_pred.flatten(),
    'Sai_so': abs(y_test_real.flatten() - y_pred.flatten())
})

result.head(20)
plt.figure(figsize=(8, 6))
plt.scatter(y_test_real, y_pred)
plt.xlabel("Giá thực tế")
plt.ylabel("Giá dự đoán")
plt.title("So sánh giá thực tế và giá dự đoán")
plt.grid(True)
plt.show()

In [ ]:
new_car = pd.DataFrame({
    'Age': [60],
    'KM': [80000],
    'Weight': [1050],
    'HP': [110],
    'MetColor': [1],
    'CC': [1600],
    'Doors': [5]
})

new_car_scaled = scaler_X.transform(new_car)

pred_scaled = model.predict(new_car_scaled)

pred_price = scaler_y.inverse_transform(pred_scaled)

print("Giá xe dự đoán:", pred_price[0][0])

Train ANN nhận diện chữ/số

Bước 1: Cài và import thư viện

In [ ]:
!pip install datasets

import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset

from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

Bước 2: Tải dữ liệu MNIST từ Hugging Face

In [ ]:
dataset = load_dataset("ylecun/mnist")

print(dataset)

Bước 3: Chuyển dữ liệu ảnh thành dạng ANN dùng được

In [ ]:
# Lấy dữ liệu train
X_train = np.array([
    np.array(img).reshape(-1)
    for img in dataset["train"]["image"]
]) / 255.0

y_train = np.array(dataset["train"]["label"])

# Lấy dữ liệu test
X_test = np.array([
    np.array(img).reshape(-1)
    for img in dataset["test"]["image"]
]) / 255.0

y_test = np.array(dataset["test"]["label"])

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Bước 4: Xem thử ảnh chữ số

In [ ]:
index = 0

plt.imshow(X_train[index].reshape(28, 28), cmap="gray")
plt.title(f"Nhãn thật: {y_train[index]}")
plt.axis("off")
plt.show()

Bước 5: Xây dựng mô hình ANN

In [ ]:
model = Sequential()

model.add(Dense(256, activation='relu', input_shape=(784,)))
model.add(Dropout(0.2))

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(64, activation='relu'))

# 10 lớp đầu ra: số 0 đến 9
model.add(Dense(10, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Bước 6: Train mô hình với EarlyStopping


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Bước 7: Vẽ biểu đồ Loss và Accuracy

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Biểu đồ Loss của ANN")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Biểu đồ Accuracy của ANN")
plt.legend()
plt.grid(True)
plt.show()

Bước 8: Đánh giá và hiển thị kết quả dự đoán

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

print("10 nhãn thật:", y_test[:10])
print("10 nhãn dự đoán:", y_pred[:10])

print(classification_report(y_test, y_pred))

Hiển thị vài ảnh dự đoán:

In [ ]:
plt.figure(figsize=(12, 8))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap="gray")
    plt.title(f"Thật: {y_test[i]} | Dự đoán: {y_pred[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()